# Replication notebook: paper figures 2, 4, 6

Reproduces the multi-panel figures from the original `visualization.ipynb`
using outputs from the `fixed/` pipeline:

* **Figure 2** — per-subject DistCorr bars (intra-subject, cross-subject,
  surrogate) for iEEG and EEG.
* **Figure 4** — effect of mask-intensity sweep (0%, 25%, 50%, 75%, 100%)
  on DistCorr, per subject.
* **Figure 6** — effect of electrode coverage (Local / Non-Local / All)
  on DistCorr, per subject.

## Prerequisites

Before running this notebook the student must have produced the
following training outputs by editing `fixed/configs/settings.yaml` and
running the listed commands. The `mask_intensity` keys mentioned below
are overridden at run-time by `main.py`; everything else is read from
the YAML.

| Source for | `dataset_type` | `mode`             | Command to run                              |
|---|---|---|---|
| Fig 2 intra-subject (iEEG) | `iEEG`             | `train`     | `python main.py` |
| Fig 2 intra-subject (EEG)  | `EEG`              | `train`     | `python main.py` |
| Fig 2 cross-subject (EEG)  | `EEG`              | `test_new`  | `python main.py` |
| Fig 2 cross-subject (iEEG) | (uses adjacency)   | —           | `python generate_adj_matrix_ieeg_unseen.py` then `python Test_new_subject_iEEG.py` |
| Fig 2 surrogate (iEEG)     | `Surrogate_iEEG`   | `train`     | `python main.py` |
| Fig 2 surrogate (EEG)      | `Surrogate_EEG`    | `train`     | `python main.py` |
| Fig 4 sweep (iEEG)         | `iEEG`             | `train`     | `python main.py` (covers all 5 intensities) |
| Fig 4 sweep (EEG)          | `EEG`              | `train`     | `python main.py` |
| Fig 6 coverage (both)      | `iEEG` / `EEG`     | —           | `python run_coverage_experiment.py` |

After each run, set `dataset_type` to the next entry in the table and
re-run the same command. `fixed/main.py` automatically sweeps all five
mask intensities.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make the fixed/ package and the notebooks helper importable.
_NB_DIR = Path.cwd()
_PKG = _NB_DIR.parent if _NB_DIR.name == 'notebooks' else _NB_DIR
for p in (str(_PKG), str(_PKG / 'notebooks')):
    if p not in sys.path:
        sys.path.insert(0, p)

from _helpers import (
    load_paths, load_dcorr, mean_var_dcorr,
)

# Resolve the package's configured `results_dir`. Pass an override here
# if you want figures generated from a results directory that differs
# from the one configured in configs/device_path.yaml.
paths = load_paths()
RESULTS_DIR = paths.results_dir

# Where to drop the generated figures.
FIG_OUT = _NB_DIR / 'figures'
FIG_OUT.mkdir(exist_ok=True)

# Run-configuration that everything else inherits from. Change these to
# match whatever you used for training.
NEIGHBORHOOD_METHOD = 'atlas'        # 'atlas' (manuscript) or 'knn'
MODEL_TYPE = 'instantaneous'         # 'instantaneous' or 'lagged'
MASK_INTENSITY_SWEEP = [0.0, 0.25, 0.5, 0.75, 1.0]

print(f'results_dir = {RESULTS_DIR}')
print(f'figures will be written to {FIG_OUT}')

## Figure 4 — mask-intensity sweep (iEEG)

For each iEEG subject we load `DCORR_<i>.npy` at every mask intensity
and plot mean ± variance with errorbars. This is the panel directly
exposing the masking knob defined in Section *Spatially Masked
Regression Model*.

In [ ]:
NUM_IEEG_SUBJECTS = 12

stats = mean_var_dcorr(
    results_dir=RESULTS_DIR,
    dataset_type='iEEG', mode='train',
    subjects=range(NUM_IEEG_SUBJECTS),
    mask_intensities=MASK_INTENSITY_SWEEP,
    neighborhood_method=NEIGHBORHOOD_METHOD,
    model_type=MODEL_TYPE,
)

intensities_pct = [m * 100 for m in MASK_INTENSITY_SWEEP]
colors = plt.cm.get_cmap('tab20', NUM_IEEG_SUBJECTS)

fig = plt.figure(figsize=(8, 4), dpi=300)
for s in range(NUM_IEEG_SUBJECTS):
    means, varis = [], []
    for m in MASK_INTENSITY_SWEEP:
        cell = stats[s][m]
        means.append(cell['mean'] if cell else np.nan)
        varis.append(cell['var'] if cell else 0.0)
    plt.errorbar(
        intensities_pct, means, yerr=varis,
        fmt='-o', capsize=2, label=f'Subject {s + 1}', color=colors(s),
    )

plt.xticks(intensities_pct, fontsize=12)
plt.xlabel('Mask Intensity', fontsize=15)
plt.ylabel('Distance Correlation', fontsize=15)
plt.legend(ncol=4, fontsize=8)
plt.grid(True)
plt.tight_layout()
fig.savefig(FIG_OUT / 'fig4_iEEG_mask_intensity.png')
fig.savefig(FIG_OUT / 'fig4_iEEG_mask_intensity.svg')
plt.show()

## Figure 4 — mask-intensity sweep (EEG)

Same plot for the EEG dataset, 15 subjects. We do not skip any subject;
if the original notebook excluded a subject, that was for cosmetic
reasons and is *not* recommended for replication.

In [ ]:
NUM_EEG_SUBJECTS = 15

stats = mean_var_dcorr(
    results_dir=RESULTS_DIR,
    dataset_type='EEG', mode='train',
    subjects=range(NUM_EEG_SUBJECTS),
    mask_intensities=MASK_INTENSITY_SWEEP,
    neighborhood_method=NEIGHBORHOOD_METHOD,
    model_type=MODEL_TYPE,
)

intensities_pct = [m * 100 for m in MASK_INTENSITY_SWEEP]
colors = plt.cm.get_cmap('tab20', NUM_EEG_SUBJECTS)

fig = plt.figure(figsize=(8, 4), dpi=300)
for s in range(NUM_EEG_SUBJECTS):
    means, varis = [], []
    for m in MASK_INTENSITY_SWEEP:
        cell = stats[s][m]
        means.append(cell['mean'] if cell else np.nan)
        varis.append(cell['var'] if cell else 0.0)
    plt.errorbar(
        intensities_pct, means, yerr=varis,
        fmt='-o', capsize=2, label=f'Subject {s + 1}', color=colors(s),
    )

plt.xticks(intensities_pct, fontsize=12)
plt.xlabel('Mask Intensity', fontsize=15)
plt.ylabel('Distance Correlation', fontsize=15)
plt.legend(ncol=5, fontsize=8)
plt.grid(True)
plt.tight_layout()
fig.savefig(FIG_OUT / 'fig4_EEG_mask_intensity.png')
fig.savefig(FIG_OUT / 'fig4_EEG_mask_intensity.svg')
plt.show()

## Figure 2 — intra / cross / surrogate per-subject bars

Three conditions per subject:

* **Intra-subject** — `dataset_type='EEG' | 'iEEG'`, `mode='train'`,
  `mask_intensity=1.0` (full local mask, matching the paper).
* **Cross-subject** — for EEG, `mode='test_new'`. For iEEG, the
  per-electrode best-donor DistCorr is read from
  `results_dir/plot/sub_<i>/best_Dcorr.npy` written by
  `Test_new_subject_iEEG.py`.
* **Surrogate** — `dataset_type='Surrogate_EEG' | 'Surrogate_iEEG'`,
  `mode='train'`, `mask_intensity=1.0` (phase-shuffled). To use the
  IAAFT or block-shuffle columns instead, change the dataset_type
  in the helper call below.

In [ ]:
def aggregate_three_conditions(num_subjects, real_dataset, surrogate_dataset,
                                cross_loader,
                                neighborhood_method=NEIGHBORHOOD_METHOD,
                                model_type=MODEL_TYPE):
    """Build a tidy DataFrame for one modality.

    Each row = (subject, Setting, mean, var).
    """
    rows = []
    for s in range(num_subjects):
        try:
            v = load_dcorr(RESULTS_DIR, real_dataset, 'train', s, 1.0,
                            neighborhood_method, model_type)
            rows.append(dict(Subject=s + 1, Setting='Intra-Subject',
                              mean=np.mean(v), var=np.var(v)))
        except FileNotFoundError as e:
            print(f'  intra: sub_{s} missing -- {e}')

        try:
            v = cross_loader(s)
            if v is not None and v.size:
                rows.append(dict(Subject=s + 1, Setting='Cross-Subject',
                                  mean=float(np.nanmean(v)),
                                  var=float(np.nanvar(v))))
        except FileNotFoundError as e:
            print(f'  cross: sub_{s} missing -- {e}')

        try:
            v = load_dcorr(RESULTS_DIR, surrogate_dataset, 'train', s, 1.0,
                            neighborhood_method, model_type)
            rows.append(dict(Subject=s + 1, Setting='Surrogate',
                              mean=np.mean(v), var=np.var(v)))
        except FileNotFoundError as e:
            print(f'  surrogate: sub_{s} missing -- {e}')

    return pd.DataFrame(rows)


def _load_eeg_cross(s):
    return load_dcorr(RESULTS_DIR, 'EEG', 'test_new', s, 1.0,
                       NEIGHBORHOOD_METHOD, MODEL_TYPE)


def _load_ieeg_cross(s):
    path = Path(RESULTS_DIR) / 'plot' / f'sub_{s}' / 'best_Dcorr.npy'
    if not path.exists():
        raise FileNotFoundError(path)
    return np.asarray(np.load(path), dtype=float)


df_eeg = aggregate_three_conditions(
    num_subjects=NUM_EEG_SUBJECTS,
    real_dataset='EEG', surrogate_dataset='Surrogate_EEG',
    cross_loader=_load_eeg_cross,
)
df_ieeg = aggregate_three_conditions(
    num_subjects=NUM_IEEG_SUBJECTS,
    real_dataset='iEEG', surrogate_dataset='Surrogate_iEEG',
    cross_loader=_load_ieeg_cross,
)

print('EEG rows:', len(df_eeg))
print('iEEG rows:', len(df_ieeg))
df_eeg.head()

In [ ]:
def plot_three_condition_bars(df, ylim, fname_base):
    if df.empty:
        print('Nothing to plot -- DataFrame is empty.')
        return
    sns.set_style('whitegrid')
    fig = plt.figure(figsize=(12, 6), dpi=300, facecolor='white')
    palette = {'Intra-Subject': 'teal',
                'Cross-Subject': 'cyan',
                'Surrogate': 'purple'}
    ax = sns.barplot(
        data=df, x='Subject', y='mean', hue='Setting',
        palette=palette, errorbar=None,
    )
    ax.set_facecolor('white')

    grouped = (df.groupby(['Subject', 'Setting'], sort=False)['var']
                  .first().reset_index())
    for bar, (_, row) in zip(ax.patches, grouped.iterrows()):
        x = bar.get_x() + bar.get_width() / 2
        y = bar.get_height()
        ax.errorbar(x=x, y=y, yerr=row['var'],
                     ecolor='black', capsize=3, fmt='none')

    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.set_xlabel('Subject', fontsize=24)
    ax.set_ylabel('Distance Correlation', fontsize=24)
    ax.tick_params(axis='both', labelsize=24)
    ax.legend(fontsize=20, ncol=3)
    ax.grid(True, which='both', linestyle='--', linewidth=0.7, alpha=0.7)
    ax.set_axisbelow(True)
    plt.ylim(*ylim)
    plt.tight_layout()
    fig.savefig(FIG_OUT / f'{fname_base}.png')
    fig.savefig(FIG_OUT / f'{fname_base}.svg')
    plt.show()


plot_three_condition_bars(df_eeg,  ylim=(0, 1.1),  fname_base='fig2_EEG_bar')
plot_three_condition_bars(df_ieeg, ylim=(0, 0.78), fname_base='fig2_iEEG_bar')

## Figure 6 — Local / Non-Local / All coverage bars

Three masking conditions per subject, produced by
`fixed/run_coverage_experiment.py`. The script writes one folder per
(subject, condition) pair under
`<results_dir>/coverage/<dataset_type>/sub_<i>/nbhd-<m>/model-<t>/cond-<c>/`.

In [ ]:
def load_coverage_dcorr(dataset_type, subject, condition):
    """Read the DCORR array for one (subject, condition) cell of the
    coverage experiment. Returns a flat 1-D numpy array, or None."""
    path = (Path(RESULTS_DIR) / 'coverage' / dataset_type / f'sub_{subject}'
            / f'nbhd-{NEIGHBORHOOD_METHOD}'
            / f'model-{MODEL_TYPE}'
            / f'cond-{condition}'
            / f'DCORR_{subject}.npy')
    if not path.exists():
        return None
    raw = np.load(path, allow_pickle=True)
    if isinstance(raw, np.ndarray) and raw.dtype == object:
        return np.concatenate([np.asarray(r).ravel() for r in raw]).astype(float)
    return np.asarray(raw, dtype=float).ravel()


def coverage_dataframe(num_subjects, dataset_type):
    rows = []
    labels = {'local': 'Local', 'non_local': 'Non-Local', 'all': 'All'}
    for s in range(num_subjects):
        for cond_key, cond_label in labels.items():
            v = load_coverage_dcorr(dataset_type, s, cond_key)
            if v is None:
                continue
            rows.append(dict(
                Subject=s + 1, Condition=cond_label,
                Score=float(np.mean(v)), STD=float(np.var(v)),
            ))
    return pd.DataFrame(rows)


df_cov_eeg = coverage_dataframe(NUM_EEG_SUBJECTS, 'EEG')
df_cov_ieeg = coverage_dataframe(NUM_IEEG_SUBJECTS, 'iEEG')
print('EEG coverage rows:', len(df_cov_eeg))
print('iEEG coverage rows:', len(df_cov_ieeg))

In [ ]:
def plot_coverage(df, ylim, fname_base):
    if df.empty:
        print(f'Skipping {fname_base}: no coverage data found.')
        return
    sns.set(style='whitegrid')
    fig = plt.figure(figsize=(12, 6), dpi=300)
    palette = {'Local': 'royalblue', 'Non-Local': 'red', 'All': 'gold'}

    ax = sns.barplot(data=df, x='Subject', y='Score', hue='Condition',
                     palette=palette, errorbar=None)
    for _, row in df.iterrows():
        hue_offset = {'Local': -0.25, 'Non-Local': 0, 'All': 0.25}[row['Condition']]
        ax.errorbar(
            x=row['Subject'] + hue_offset - 1,
            y=row['Score'], yerr=row['STD'],
            fmt='none', c='black', capsize=3,
        )
    ax.set_xlabel('Subject', fontsize=24)
    ax.set_ylabel('Distance Correlation', fontsize=24)
    ax.tick_params(axis='both', labelsize=24)
    ax.legend(fontsize=20, ncol=3, loc='upper right')
    ax.set_ylim(*ylim)
    sns.despine()
    ax.grid(True, which='both', linewidth=0.7, alpha=0.7)
    ax.set_axisbelow(True)
    plt.tight_layout()
    fig.savefig(FIG_OUT / f'{fname_base}.png')
    fig.savefig(FIG_OUT / f'{fname_base}.svg')
    plt.show()


plot_coverage(df_cov_eeg,  ylim=(0.8, 1.0),  fname_base='fig6_EEG_coverage')
plot_coverage(df_cov_ieeg, ylim=(0.3, 0.85), fname_base='fig6_iEEG_coverage')

All figures written into `figures/`. Each panel is saved as both `.png`
and `.svg` so it can be edited in Inkscape / Illustrator if you want to
polish the layout for the final manuscript draft.